In [ ]:
from pathlib import Path
import torch.nn.functional as F
import numpy as np
from tqdm import tqdm
from collections import defaultdict
from torch.utils.data import DataLoader, random_split
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from huggingface_hub import hf_hub_download
import json, torch
from scipy.sparse import csr_matrix
from sklearn.metrics import f1_score, precision_score, recall_score, precision_recall_fscore_support
from google.colab import userdata
from huggingface_hub import HfApi
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
def subcluster_margin_loss_per_top(sub_logits, sub_targets, top_targets, top_to_sub_mask, margin=0.3, top_k=3):
    """ Hierarchy-correct margin loss.
    Enforces margin ONLY between sibling subs under the SAME top.
    Skips tops with <2 subclusters. """
    B, _ = sub_logits.shape
    T = top_targets.shape[1]

    loss_terms = []
    for t in range(T):  # T is small (~9)
        # Check which samples activate this top
        active_samples = top_targets[:, t] == 1
        if not active_samples.any():
            continue

        # Subclusters belonging to this top
        sub_mask = top_to_sub_mask[t].bool()  # [S]

        # skip tops with fewer than 2 subclusters
        if sub_mask.sum() < 2:
            continue

        # Slice batch for this top
        logits_t = sub_logits[active_samples][:, sub_mask]    # [B_t, S_t]
        targets_t = sub_targets[active_samples][:, sub_mask]  # [B_t, S_t]

        pos_mask = targets_t == 1
        neg_mask = targets_t == 0

        # Need at least one positive and one negative
        if not pos_mask.any() or not neg_mask.any():
            continue

        # Mask logits
        pos_logits = logits_t.masked_fill(~pos_mask, float("-inf"))
        neg_logits = logits_t.masked_fill(~neg_mask, float("-inf"))

        # Aggregate positives (max is fine here)
        pos_logit = pos_logits.max(dim=1).values  # [B_t]

        # Hard negatives
        hard_neg = torch.topk(neg_logits, k=min(top_k, neg_logits.shape[1]),mdim=1).values  # [B_t, K]

        # Margin violation
        violation = torch.relu(margin - (pos_logit.unsqueeze(1) - hard_neg))

        loss_terms.append(violation.mean())

    if len(loss_terms) == 0:
        return sub_logits.new_tensor(0.0)

    return torch.stack(loss_terms).mean()

In [ ]:
def weighted_focal_loss(inputs: torch.Tensor, targets: torch.Tensor, weights: torch.Tensor = None,
                        alpha: float = 0.25, gamma: float = 1.5, eps: float = 1e-8):

    p = torch.sigmoid(inputs)
    p = p.clamp(min=eps, max=1. - eps)

    # Compute the focal modulation for each element
    ce_loss = - (targets * torch.log(p) + (1 - targets) * torch.log(1 - p))
    pt = targets * p + (1 - targets) * (1 - p)  # p_t term
    focal_factor = (1 - pt) ** gamma
    # Apply alpha-balancing
    alpha_factor = targets * alpha + (1 - targets) * (1 - alpha)
    # Combine everything
    loss = alpha_factor * focal_factor * ce_loss
    # Apply class/sample weighting if provided
    if weights is not None:
        loss = loss * weights
    count_nonzero = torch.count_nonzero(weights)
    #return loss.mean()
    return loss.sum() / (count_nonzero + 1e-6)

In [ ]:
class AspectAttention(nn.Module):
    def __init__(self, hidden_dim, aspect_emb_dim, d_k=256, d_v=256):
        super().__init__()
        self.Wq = nn.Linear(aspect_emb_dim, d_k, bias=False)
        self.Wk = nn.Linear(hidden_dim, d_k, bias=False)
        self.Wv = nn.Linear(hidden_dim, d_v, bias=False)
        self.Wo = nn.Linear(d_v, hidden_dim)

    def forward(self, enc, aspect_emb, attention_mask):
        """
        enc: (B, L, H)
        aspect_emb: (B, N, E)  <-- Now expects 3D input (Standard)
        """
        # Linear layers handle the (B, N) dimensions automatically
        Q = self.Wq(aspect_emb)          # (B, N, d_k)
        K = self.Wk(enc)                 # (B, L, d_k)
        V = self.Wv(enc)                 # (B, L, d_v)

        # Compute Scores: (B, N, d_k) x (B, d_k, L) -> (B, N, L)
        scores = torch.matmul(Q, K.transpose(1, 2)) / (Q.size(-1) ** 0.5)

        # Masking
        mask = attention_mask.squeeze(-1).unsqueeze(1) # (B, 1, L)
        scores = scores.masked_fill(mask == 0, -1e4) # Safe -1e4
        attn = F.softmax(scores, dim=-1)
        context = torch.matmul(attn, V)  # (B, N, d_v)

        return self.Wo(context)   #(B, N, H)

In [ ]:
class OrthogonalityRegularizer:
    def __init__(self, lambda_ortho=0.1):
        """
        lambda_ortho: Weight of this loss (usually 0.01 to 0.1)
        """
        self.lambda_ortho = lambda_ortho

    def compute_loss(self, query_vectors):
        """
        query_vectors shape: (1, num_labels, hidden_dim) or (num_labels, hidden_dim)
        """
        q = query_vectors.squeeze(0)
        q_norm = F.normalize(q, p=2, dim=1)
        gram_matrix = torch.mm(q_norm, q_norm.t())

        eye = torch.eye(q.shape[0], device=q.device)
        
        # (We actually don't care about the diagonal being 1, we imply it,
        # so we mask the diagonal to focus purely on making distinct heads separate)
        ortho_loss = ((gram_matrix * (1 - eye)) ** 2).mean()

        return self.lambda_ortho * ortho_loss

In [ ]:
def visualize_label_distinctness(model, label_names=None):
    """
    Plots the Cosine Similarity Matrix of the model's learned query vectors.
    Args:
        model: Your PyTorch model (must have model.head_query)
        label_names: List of strings (e.g., ['Price', 'Battery', 'Screen'])
    """
    with torch.no_grad():
        q1 = model.head_query.detach().cpu()
        q2 = model.sub_query.detach().cpu()
        if q1.dim() == 3:
            q1 = q1.squeeze(0)
        if q2.dim() == 3:
            q2 = q2.squeeze(0)

    q1_norm = F.normalize(q1, p=2, dim=1)
    q2_norm = F.normalize(q2, p=2, dim=1)

    # Compute Similarity Matrix (Dot product of normalized vectors)
    # Shape: (Num_Labels, Num_Labels)
    similarity_matrix1 = torch.mm(q1_norm, q1_norm.t()).numpy()
    similarity_matrix2 = torch.mm(q2_norm, q2_norm.t()).numpy()

    avg_off_diag1 = (np.sum(np.abs(similarity_matrix1)) - len(similarity_matrix1)) / (len(similarity_matrix1)**2 - len(similarity_matrix1))
    avg_off_diag2 = (np.sum(np.abs(similarity_matrix2)) - len(similarity_matrix2)) / (len(similarity_matrix2)**2 - len(similarity_matrix2))
    print(f"Average Off-Diagonal Similarity: {avg_off_diag1:.8f}")
    print(f"Average Off-Diagonal Similarity: {avg_off_diag2:.8f}")
    if avg_off_diag1 > 0.5 and avg_off_diag2 > 0.5:
        print("WARNING: High overlap detected. Your aspect heads are collapsing!")
    else:
        print("SUCCESS: Vectors are distinct.")

In [ ]:
def load_tokenizer_and_model_from_hf(cfg, top_dim, sub_dim, device):
    hf_repo_id = cfg.hf_repo
    hf_subfolder = cfg.hf_encoder_subfolder
    checkpoint_filename = "final_stage1_v6/best_stage1.pt"

    print(f" Loading Tokenizer & Base Architecture from: {hf_repo_id} (subfolder: {hf_subfolder})")
    try:
        tokenizer = AutoTokenizer.from_pretrained(hf_repo_id, subfolder=hf_subfolder)
        print(f" Tokenizer loaded.")
    except Exception as e:
        print(f" Failed to load tokenizer: {e}")
        tokenizer = AutoTokenizer.from_pretrained(hf_repo_id)

    print(f" Initializing Architecture...")
    model = HierarchicalClassifier(
        hf_repo_id,
        subfolder=hf_subfolder,
        top_dim=top_dim,
        sub_dim=sub_dim,
        dropout=0.2
    ).to(device)

    print(f"⬇️ Downloading weights from: {hf_repo_id}/{checkpoint_filename} ...")
    try:
        cached_path = hf_hub_download(
            repo_id=hf_repo_id,
            filename=checkpoint_filename)

        checkpoint = torch.load(cached_path, map_location=device)
        if "model_state_dict" in checkpoint:
            state_dict = checkpoint["model_state_dict"]
        else:
            state_dict = checkpoint 
        missing, unexpected = model.load_state_dict(state_dict, strict=False)

        print(f"✅ Trained weights loaded successfully!")
        if missing: print(f"   Missing keys: {len(missing)} (Ensure this matches expectation)")
        if unexpected: print(f"   Unexpected keys: {len(unexpected)}", unexpected)

    except Exception as e:
        print(f"❌ Critical Error: Could not load checkpoint from Hugging Face.")
        print(f"   Error details: {e}")
        print("   Returning model with RANDOM weights (Architecture only).")

    return tokenizer, model

In [ ]:
def class_wise_eval(y_true, y_pred_probs):
    n_classes = y_true.shape[1]
    print("-" * 85)
    print(f"{'Class ID':<10} | {'Threshold':<10} | {'F1':<10} | {'Precision':<10} | {'Recall':<10} | {'Support':<10}")
    print("-" * 85)
    final_avg_thresholds = [0.5]*n_classes
    class_metrics = {}
    macro_f1_scores = []

    for c in range(n_classes):
        preds_bin = (y_pred_probs[:, c] > final_avg_thresholds[c]).astype(int)
        true_bin = y_true[:, c]

        # Calculate metrics
        precision = precision_score(true_bin, preds_bin, zero_division=0)
        recall = recall_score(true_bin, preds_bin, zero_division=0)
        f1 = f1_score(true_bin, preds_bin, zero_division=0)
        support = true_bin.sum()

        macro_f1_scores.append(f1)

        # Store
        class_metrics[c] = {
            "threshold": final_avg_thresholds[c],
            "f1": f1,
            "precision": precision,
            "recall": recall,
            "support": support
        }

        # Print Row
        print(f"{c} | {final_avg_thresholds[c]:.3f}      | {f1:.4f}     | {precision:.4f}     | {recall:.4f}     | {int(support)}")

    print("-" * 85)
    avg_f1 = np.mean(macro_f1_scores)
    print(f" Final Macro F1 Score: {avg_f1:.4f}")

    return final_avg_thresholds, avg_f1, class_metrics

In [ ]:
class Config:
    output_dir = Path(r"/content/stage2_joint") 
    hf_repo = "Faisal191/aspect-classifier" 
    hf_encoder_subfolder = "Domain_trained_encoder"     
    hf_compact_path_in_repo= "stage2_full_checkpoint/latest_checkpoint.pt" 
    data_path = Path(r"/content/final_aspa_data_hierarchical_with_sentiments_temp_v6.json")
    epochs = 15
    batch_size = 16
    lr_encoder = 1e-5
    lr_heads = 1e-4
    max_len = 256
    val_split = 0.1
    use_amp = True
    unfreeze_full_epoch = 3
    use_sampler = False
    top_k_train = 3
    top_k_eval = 3
    prob_transfer_weight = 0.4
    gating_mode = "add"
    margin_start_epoch = 6
    clip_grad_norm = 1.0
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    seed = 42
    USE_HIER_MASK_IN_EVAL = True  # apply hierarchical mask to sub predictions at eval/inference

cfg = Config()
cfg.output_dir.mkdir(parents=True, exist_ok=True)

torch.manual_seed(cfg.seed)
np.random.seed(cfg.seed)
torch.backends.cudnn.benchmark = True
DEVICE = cfg.device
print("Device:", DEVICE)


In [ ]:
def csr_to_torch_sparse(csr):
    coo = csr.tocoo()
    if coo.nnz == 0:
        indices = torch.empty((2,0), dtype=torch.long)
        values = torch.empty((0,), dtype=torch.float32)
    else:
        indices = torch.tensor(np.array([coo.row, coo.col]), dtype=torch.long)
        values = torch.tensor(coo.data, dtype=torch.float32)
    return torch.sparse_coo_tensor(indices, values, torch.Size(coo.shape))

def compute_metrics_numpy(y_true, y_pred, threshold=0.5):
    y_pred_bin = (y_pred > threshold).astype(int)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred_bin, average="macro", zero_division=0
    )
    return precision, recall, f1

In [ ]:
class HierarchicalDataset(torch.utils.data.Dataset):
    def __init__(self, data, tokenizer, max_len=256):
        self.data = data
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        text = item.get("text","")
        enc = self.tokenizer(text, truncation=True, padding="max_length",
                             max_length=self.max_len, return_tensors="pt")
        enc = {k:v.squeeze(0) for k,v in enc.items()}
        top_ids = np.array(item["top_cluster_ids"], dtype=np.float32)
        sub_ids = np.array(item["sub_cluster_ids"], dtype=np.float32)
        sub_sparse = csr_to_torch_sparse(csr_matrix(sub_ids.reshape(1,-1)))
        return {
            "input_ids": enc["input_ids"],
            "attention_mask": enc["attention_mask"],
            "top_labels": torch.tensor(top_ids, dtype=torch.float32),
            "sub_labels_sparse": sub_sparse
        }

def collate_fn(batch):
    return {
        "input_ids": torch.stack([b["input_ids"] for b in batch]),
        "attention_mask": torch.stack([b["attention_mask"] for b in batch]),
        "top_labels": torch.stack([b["top_labels"] for b in batch]),
        "sub_labels_sparse": [b["sub_labels_sparse"] for b in batch]
    }

In [ ]:
class HierarchicalClassifier(nn.Module):
    def __init__(self, encoder_name, top_dim, sub_dim, dropout=0.2, subfolder=None):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(encoder_name, subfolder=subfolder)
        self.thresholds = torch.tensor([0.5]*9, dtype=torch.float32)
        h_dim = self.encoder.config.hidden_size
        self.top_dim = top_dim
        self.sub_dim = sub_dim
        self.aspect_emb_dim = h_dim
        self.head_aspect_attention = AspectAttention(
            hidden_dim=h_dim, aspect_emb_dim=self.aspect_emb_dim)
        self.head_query = nn.Parameter(torch.empty(1, top_dim, h_dim))
        nn.init.normal_(self.head_query, mean=0, std=0.02)
        self.head_gate_proj = nn.Linear(h_dim * 2, h_dim)
        self.head_norm = nn.LayerNorm(h_dim)
        self.head_weight = nn.Parameter(torch.randn(top_dim, h_dim))
        self.head_bias = nn.Parameter(torch.zeros(top_dim))

        torch.nn.init.normal_(self.head_weight, std=0.02)
        self.head_dropout = nn.Dropout(dropout)

        self.sub_aspect_attention = AspectAttention(
            hidden_dim=h_dim, aspect_emb_dim=self.aspect_emb_dim)
        self.sub_query = nn.Parameter(torch.empty(1, sub_dim, h_dim))
        nn.init.normal_(self.sub_query, mean=0, std=0.02)
        self.sub_gate_proj = nn.Linear(h_dim * 2, h_dim)

        self.sub_norm = nn.LayerNorm(h_dim)
        self.sub_weight = nn.Parameter(torch.randn(sub_dim, h_dim))
        self.sub_bias = nn.Parameter(torch.zeros(sub_dim))

        torch.nn.init.normal_(self.sub_weight, std=0.02)
        self.sub_dropout = nn.Dropout(dropout)

        self.top_to_sub_map = None          # sparse tensor (top_dim × sub_dim)
        self.top_to_sub_dense = None        # dense cache for matmul
        self.top_dim = top_dim
        self.sub_dim = sub_dim
        self.top_pos_weight = None
        self.sub_pos_weight = None
        self.prob_weight = nn.Parameter(torch.full((1,), 0.4))

    def set_top_to_sub_map(self, mapping: dict):
        self.top_to_sub_dict = mapping

        rows, cols = [], []
        for t, subs in mapping.items():
            rows.extend([t] * len(subs))
            cols.extend(subs)

        indices = torch.tensor([rows, cols], dtype=torch.long)
        values = torch.ones(len(rows), dtype=torch.float32)
        sparse_map = torch.sparse_coo_tensor(indices, values, (self.top_dim, self.sub_dim))
        self.top_to_sub_map = sparse_map.coalesce()
        self.top_to_sub_dense = None


    def _get_dense_map(self, device, dtype):
        """
        Safely gets (or builds) dense mapping tensor.
        - Auto moves to correct device/dtype (handles AMP).
        - Caches result for reuse.
        """
        if self.top_to_sub_dense is None or self.top_to_sub_dense.device != device:
            dense_map = self.top_to_sub_map.to_dense().to(device)
            self.top_to_sub_dense = dense_map
        # ensure dtype matches current autocast precision
        if self.top_to_sub_dense.dtype != dtype:
            self.top_to_sub_dense = self.top_to_sub_dense.to(dtype)
        return self.top_to_sub_dense

    def forward(self, input_ids, attention_mask, gating_mode="add", prob_weight=0.5):
        # Encoder
        enc = self.encoder(input_ids, attention_mask=attention_mask).last_hidden_state
        batch_size = enc.size(0)
        mask = attention_mask.unsqueeze(-1).float() #(B, L, 1)
        pooled = (enc * mask).sum(1) / mask.sum(1).clamp(min=1.0)
        top_query_expanded = self.head_query.expand(batch_size, -1, -1) # (B, N, H)
        aspect_context = self.head_aspect_attention(enc, top_query_expanded, mask) # (B, N, H)
        pooled_expanded = pooled.unsqueeze(1).expand(-1, self.top_dim, -1) # (B, N, H)
        # Gating Logic
        # Concatenate: (B, N, 2*H)
        combined = torch.cat([pooled_expanded, aspect_context], dim=2)
        # Compute Gate Alpha: (B, N, H)
        alpha = torch.sigmoid(self.head_gate_proj(combined))

        # Fuse: Alpha controls how much we use Pooled vs Attention
        fused_embedding = alpha * pooled_expanded + (1 - alpha) * aspect_context
        fused_norm = self.head_norm(self.head_dropout(fused_embedding))

        # 4. Top-Level Prediction
        top_logits = (fused_norm * self.head_weight).sum(dim=-1) + self.head_bias
        # Now opened for sub
        p_top = torch.sigmoid(top_logits)

        # Compute sub-cluster prior
        batch_size, device, dtype = p_top.size(0), p_top.device, p_top.dtype
        if self.top_to_sub_map is not None:
            map_dense = self._get_dense_map(device, dtype)
            sub_prior = torch.matmul(p_top, map_dense)
            sub_prior.clamp_(0.0, 1.0)
        else:
            sub_prior = torch.zeros(batch_size, self.sub_dim, device=device, dtype=dtype)

        # Move thresholds to the same device as p_top
        sub_query_expanded = self.sub_query.expand(batch_size, -1, -1) # (B, N, H)
        sub_aspect_context = self.sub_aspect_attention(enc, sub_query_expanded, mask) # (B, N, H)
        # Gating Logic
        # Concatenate: (B, N, 2*H)
        pooled_expanded = pooled.unsqueeze(1).expand(-1, self.sub_dim, -1)
        combined = torch.cat([pooled_expanded, sub_aspect_context], dim=2)
        # Compute Gate Alpha: (B, N, H)
        alpha = torch.sigmoid(self.sub_gate_proj(combined))

        # Fuse: Alpha controls how much we use Pooled vs Attention
        fused_embedding = alpha * pooled_expanded + (1 - alpha) * sub_aspect_context
        fused_norm = self.sub_norm(self.sub_dropout(fused_embedding))

        # 4. Top-Level Prediction
        sub_logits = (fused_norm * self.sub_weight).sum(dim=-1) + self.sub_bias

        # Hierarchical gating
        if gating_mode == "add":
            sub_logits = sub_logits + prob_weight * sub_prior
        elif gating_mode == "mul":
            sub_logits = sub_logits * (1.0 + self.prob_weight.to(p_top.device) * sub_prior.detach())

        return top_logits, sub_logits

    def compute_loss(self, epoch, cfg, top_logits, y_top, sub_logits, y_sub_sparse_list):
        alpha, beta = 0.2, 0.8
        device = sub_logits.device
        y_top = y_top.to(dtype=torch.float32, device=device)
        top_loss = F.binary_cross_entropy_with_logits(top_logits, y_top, pos_weight=self.top_pos_weight.to(device) if self.top_pos_weight is not None else None)
        ortho_calculator = OrthogonalityRegularizer(lambda_ortho=0.00001)
        # self.head_query is your learned parameter (N, 768)
        aux_loss_head = ortho_calculator.compute_loss(self.head_query)
        y_sub_dense = torch.stack([s.to_dense().squeeze(0).to(device) for s in y_sub_sparse_list])

        device, dtype = y_top.device, y_top.dtype
        if self.top_to_sub_map is not None:
            map_dense = self._get_dense_map(device, dtype)

        with torch.no_grad():
            mask = torch.zeros_like(y_sub_dense)
            valid_sub_mask = torch.matmul(y_top, map_dense)
            mask = (valid_sub_mask > 0).float()
            no_top_prediction = (mask.sum(dim=1) == 0)
            if no_top_prediction.any():
                mask[no_top_prediction] = 1.0

            if mask.sum() == 0: mask = torch.ones_like(mask)

        if self.sub_pos_weight is not None:
            pos_w = self.sub_pos_weight.to(device).unsqueeze(0).repeat(y_sub_dense.size(0),1)
            sub_weights = pos_w * mask
        else:
            sub_weights = mask

        sub_loss = weighted_focal_loss(sub_logits, y_sub_dense, weights=sub_weights)
        aux_loss_sub = ortho_calculator.compute_loss(self.sub_query)

        total_loss = alpha*top_loss + beta*sub_loss + aux_loss_head + aux_loss_sub
        top_loss = top_loss.detach()
        sub_loss = sub_loss.detach()
        margin_loss = 0

        if epoch >= cfg.margin_start_epoch:
            margin_loss = subcluster_margin_loss_per_top(sub_logits=sub_logits, sub_targets=y_sub_dense,
                top_targets=y_top, top_to_sub_mask=map_dense, margin=0.3, top_k=3)
            total_loss = total_loss + 0.05 * margin_loss
        return total_loss, top_loss, sub_loss, margin_loss

In [ ]:
def apply_hierarchical_mask_to_sub_probs(p_top, p_sub, top_thresholds, top_to_sub_map):
    """  p_top: (N, T) numpy array of top probabilities
    p_sub: (N, S) numpy array of sub probabilities
    top_thresholds: array-like length T
    top_to_sub_map: one of:
        - dict: {top_idx: [sub_idx, ...]}
        - torch.sparse_coo_tensor or torch.Tensor (dense)
        - scipy.sparse matrix
        - numpy.ndarray (dense)
    Returns: p_sub masked (numpy array)  """

    if top_to_sub_map is None or top_thresholds is None:
        return p_sub

    top_thresholds = np.array(top_thresholds)
    top_pred_bin = (p_top > top_thresholds[None, :]).astype(np.int32)  # (N, T)

    # Sanity: ensure shapes align
    T = top_pred_bin.shape[1]
    S = p_sub.shape[1]
    if dense_map.shape != (T, S):
        # try to transpose if user stored (S, T)
        if dense_map.shape == (S, T):
            dense_map = dense_map.T
        else:
            raise ValueError(f"top_to_sub_map shape mismatch: expected ({T},{S}), got {dense_map.shape}")

    mask_counts = top_pred_bin.dot(dense_map)  # int counts
    mask = (mask_counts > 0).astype(float)     # 1.0 for allowed subs, 0.0 otherwise

    # If no top predicted for an example, choose fallback: allow all subs (original behavior)
    # (original code set mask[i,:]=1.0 if relevant_subs empty)
    no_top = (top_pred_bin.sum(axis=1) == 0)
    if no_top.any():
        mask[no_top, :] = 1.0

    return p_sub * mask

In [ ]:
def evaluate(model, dataloader, device, epoch, sub_threshold=0.5, use_hier_mask=True):
    model.eval()
    all_y_top, all_p_top, all_y_sub, all_p_sub = [], [], [], []
    all_p_top_cont, all_p_sub_cont = [], []
    with torch.no_grad():
        for batch in tqdm(dataloader, desc=f"Eval E{epoch}"):
            ids, mask = batch["input_ids"].to(device), batch["attention_mask"].to(device)
            y_top = batch["top_labels"].cpu().numpy()
            y_sub = np.stack([s.to_dense().squeeze(0).cpu().numpy() for s in batch["sub_labels_sparse"]])
            top_logits, sub_logits = model(ids, mask)
            p_top = torch.sigmoid(top_logits).cpu().numpy()
            p_sub = torch.sigmoid(sub_logits).cpu().numpy()
            top_thresholds = model.thresholds.cpu().numpy()
            # apply hierarchical mask if requested (use continuous probs)
            if use_hier_mask and top_thresholds is not None:
                p_sub_masked = apply_hierarchical_mask_to_sub_probs(p_top, p_sub, top_thresholds, model._get_dense_map())
            else:
                p_sub_masked = p_sub

            if top_thresholds is not None:
                p_top_bin = (p_top > np.array(top_thresholds)).astype(int)
            else:
                p_top_bin = (p_top > 0.5).astype(int)

            p_sub_bin = (p_sub_masked > sub_threshold).astype(int)

            all_y_top.append(y_top); all_p_top.append(p_top_bin)
            all_y_sub.append(y_sub); all_p_sub.append(p_sub_bin)
            all_p_top_cont.append(p_top); all_p_sub_cont.append(p_sub_masked)

    y_top = np.vstack(all_y_top); p_top = np.vstack(all_p_top); p_top_cont = np.vstack(all_p_top_cont)
    y_sub = np.vstack(all_y_sub); p_sub = np.vstack(all_p_sub); p_sub_cont = np.vstack(all_p_sub_cont)
    _, _, f1_top = compute_metrics_numpy(y_top, p_top)
    #p_sub_th, r_sub_th, f1_sub = compute_metrics_numpy(y_sub, p_sub)
    _, sub_f1, _ = class_wise_eval(y_sub, p_sub_cont)
 

    hier_acc = np.mean((((y_top * (p_top_cont > top_thresholds)).sum(1) > 0) & ((y_sub * (p_sub_cont > 0.5)).sum(1) > 0)))
    return {"f1_top": f1_top, "sub_f1": sub_f1, "hier_acc": hier_acc}

In [ ]:
def train(cfg):
    data = json.load(open(cfg.data_path))
    top_to_sub_map = defaultdict(set)
    for it in data:
        for k,v in it.get("top_to_sub_ids", {}).items():
            top_to_sub_map[int(k)].update(v)
    top_to_sub_map = {k:list(v) for k,v in top_to_sub_map.items()}

    TOP, SUB = len(data[0]["top_cluster_ids"]), len(data[0]["sub_cluster_ids"])
    tokenizer, model = load_tokenizer_and_model_from_hf(cfg, TOP, SUB, DEVICE)
    dataset = HierarchicalDataset(data, tokenizer, max_len=cfg.max_len)

    valid_indices = np.load("/content/valid_indices.npy")
    all_indices = np.arange(len(dataset))
    train_indices = np.setdiff1d(all_indices, valid_indices)
    train_data = torch.utils.data.Subset(dataset, train_indices)
    val_data = torch.utils.data.Subset(dataset, valid_indices)
    train_loader = DataLoader(train_data, batch_size=cfg.batch_size, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_data, batch_size=cfg.batch_size, shuffle=False, collate_fn=collate_fn)

    # Model init
    model.set_top_to_sub_map(top_to_sub_map)

    # Compute lablel class weights to handle class imbalance
    top_counts = np.sum([it["top_cluster_ids"] for it in data], axis=0)
    sub_counts = np.sum([it["sub_cluster_ids"] for it in data], axis=0)
    model.top_pos_weight = torch.tensor((len(data)-top_counts)/(top_counts+1e-6),dtype=torch.float32,device=DEVICE)
    model.sub_pos_weight = torch.tensor((len(data)-sub_counts)/(sub_counts+1e-6),dtype=torch.float32,device=DEVICE)

    for name, p in model.named_parameters():
        p.requires_grad = False # Default freeze
        if "sub" in name:
            p.requires_grad = True # Train New Head

    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=cfg.lr_heads, # High LR (e.g. 1e-4)
        weight_decay=0.01)

    total_steps = len(train_loader)*cfg.epochs
    scheduler = get_linear_schedule_with_warmup(optimizer, int(0.1*total_steps), total_steps)
    scaler = torch.amp.GradScaler(enabled=cfg.use_amp)

    start_epoch, best_sum = 1, 0.0

    for epoch in range(start_epoch, cfg.epochs+1):
        if epoch == cfg.unfreeze_full_epoch:
            for p in model.encoder.parameters(): p.requires_grad=True
            print(f"Unfroze full encoder at epoch {epoch}.")
            encoder_params = []
            top_params = []
            sub_params = []

            for name, param in model.named_parameters():
                if not param.requires_grad:
                    if "head" in name.lower():
                        param.requires_grad = True
                    else:
                        continue
                if "encoder" in name:
                    encoder_params.append(param)
                elif "head" in name:
                    top_params.append(param)
                elif "sub" in name:
                    sub_params.append(param)

            optimizer = torch.optim.AdamW([
                {"params": encoder_params,   "lr": cfg.lr_encoder}, # Slower learning for base
                {"params": top_params,   "lr": cfg.lr_encoder},
                {"params": sub_params,  "lr": cfg.lr_heads} ,   # Faster learning for new layers
              ], weight_decay=0.01)
            steps_left = len(train_loader) * (cfg.epochs - epoch + 1)
            scheduler = get_linear_schedule_with_warmup(optimizer, int(0.1*steps_left), steps_left)

        model.train()
        for batch in tqdm(train_loader, desc=f"Train E{epoch}"):
            ids, mask = batch["input_ids"].to(DEVICE), batch["attention_mask"].to(DEVICE)
            y_top, y_sub_sparse = batch["top_labels"].to(DEVICE), batch["sub_labels_sparse"]

            with torch.amp.autocast(device_type='cuda', enabled=cfg.use_amp):
                top_logits, sub_logits = model(ids, mask, cfg.gating_mode, cfg.prob_transfer_weight)
                loss, top_loss, sub_loss, margin_loss = model.compute_loss(epoch, cfg, top_logits, y_top, sub_logits, y_sub_sparse, hierarchical_reg_weight=0.08)

            if not torch.isfinite(loss):
                print(f" Skipping batch due to non-finite loss at loss {loss}")
                optimizer.zero_grad()
                continue

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.clip_grad_norm)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            scheduler.step()

        # Eval
        print(f"Epoch {epoch}")
        metrics = evaluate(model, val_loader, DEVICE, epoch, use_hier_mask=cfg.USE_HIER_MASK_IN_EVAL)
        print("Hierarchy_accuracy", metrics["hier_acc"])
        print("top_f1 score", metrics["f1_top"])
        print("top_loss", top_loss)
        print("sub_loss", sub_loss)
        print("margin_loss", margin_loss)

        # Save best by F1 sum
        f1_sum = metrics["f1_top"] + metrics["sub_f1"]
        if f1_sum > best_sum:
            best_sum = f1_sum
            torch.save(model.state_dict(), cfg.output_dir/"best_stage2.pt")
            print("💾 Saved new best model (F1 sum improved).")
        visualize_label_distinctness(model)
    print(f" Training finished. Best combined F1: {best_sum:.4f}")

if __name__=="__main__":
    train(cfg)


In [ ]:
hf_token = userdata.get('HF_TOKEN')

# Initialize HfApi with the token
api = HfApi(token=hf_token)

username = "Faisal191"
repo_name = "aspect-classifier"
repo_id = f"{username}/{repo_name}"

checkpoint_path = Path(r"/content/stage2_joint/best_stage2.pt")

# Attempt the upload again with proper authentication
api.upload_file(
    path_or_fileobj=checkpoint_path,
    path_in_repo="final_stage2_v6/best_stage2.pt",
    repo_id=repo_id,
    commit_message="Upload final checkpoint")